# Lab 6 — Optimise Murshid

*Day 3, hour 5 · 50 minutes · pairs, then a leaderboard*

::: {.callout-note appearance="simple"}
**Objective** — meter the real cost and latency, then cut both with prompt caching,
a two-tier response cache and a routing table — proving after **every** step that
the evaluation suite stays green.

**Before you start** — Module 5's lab complete, with a baseline committed. Redis
from the compose stack. `data/replay_200.jsonl` — 200 conversations, intent-weighted
70/25/5.

**You finish with** — a before/after table where every row carries its eval verdict
beside its saving.
:::

::: {.callout-important}
## The rule, before you touch anything

> Never trade quality you aren't measuring for cost you are.

The leaderboard at the end ranks the **cheapest green** Murshid. A red suite is
disqualified regardless of cost, however good the number beside it looks.
:::

Each replay below takes about a minute and a half. That is the honest cost of
measuring before optimising.

In [1]:
import os, pathlib, sys, re, subprocess, urllib.request, json

# pytest and ruff colour their output; those escapes render as noise once the
# notebook is published, so they come off here rather than per command.
ANSI = re.compile(chr(27) + r"\[[0-9;]*m")

for cand in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (cand / "src" / "murshid").is_dir():
        os.chdir(cand); break
    if (cand / "murshid" / "src" / "murshid").is_dir():
        os.chdir(cand / "murshid"); break

sys.path.insert(0, "src")
os.environ["PYTHONUTF8"] = "1"
os.environ.setdefault("PYTHONPATH", "src")

def run(*args, quiet_logs=True, may_fail=False):
    """Run a course command and print what it printed.

    quiet_logs drops the structured log lines so the boxed summary is readable;
    pass quiet_logs=False when the log IS the lesson.

    may_fail=True for the commands whose job is to exit non-zero: the gate when
    it blocks, and the uncalibrated judge. Everywhere else a non-zero exit stops
    the notebook, because a traceback printed into a page that still reports as
    executed is worse than no output at all.
    """
    out = subprocess.run([sys.executable, *args], capture_output=True, text=True,
                         encoding="utf-8", errors="replace")
    text = ANSI.sub("", out.stdout + out.stderr)
    if quiet_logs:
        # Structured logs come in two shapes — the console format on a laptop and
        # JSON lines in the container — so drop both, rather than whichever one
        # the machine that built this notebook happened to emit.
        def _is_log(line):
            if line.startswith("20") and "[" in line[:40]:
                return True
            return line.lstrip().startswith('{"') and (
                '"stage"' in line or '"event"' in line or '"logger"' in line)
        text = "\n".join(l for l in text.splitlines() if not _is_log(l))
    else:
        # The log is the lesson here, but not all of it: assistant_built and the
        # per-call llm_cost records are plumbing, and they are also the widest
        # lines on the page. Keep the retries, the failover and the refusals.
        NOISE = ("llm_cost", "assistant_built")
        text = "\n".join(l for l in text.splitlines()
                          if not any(n in l for n in NOISE))
    print(text.strip())
    if out.returncode and not may_fail:
        # A failing subprocess does not fail the notebook on its own, so say so
        # loudly. Without this a broken command is a traceback in the middle of a
        # page that still reports as executed cleanly.
        raise SystemExit(f"command failed with exit code {out.returncode}: {' '.join(args)}")
    return out.returncode

# The gateway is 127.0.0.1 on a laptop and `gateway` inside compose, so take it
# from the same environment variable the application routes through rather than
# hardcoding a host that is only right in one of the two places.
GATEWAY = os.environ.get("MURSHID_PRIMARY_BASE_URL", "http://127.0.0.1:8080/v1")
GATEWAY = GATEWAY.rsplit("/v1", 1)[0].rstrip("/")

# demo_v0.py is deliberately naive — hardcoded model, inline prompt, no timeout —
# but it does read OPENAI_BASE_URL, and its default is only right on a laptop.
# Point it at the same gateway as everything else so the lab works in both places.
os.environ.setdefault("OPENAI_BASE_URL", GATEWAY + "/v1")

def fault(payload):
    """Fault injection on the course gateway: the 429 storm and the outage drill."""
    req = urllib.request.Request(
        GATEWAY + "/admin/fault", method="POST",
        data=json.dumps(payload).encode(), headers={"content-type": "application/json"})
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.load(r)

def gateway_stats():
    with urllib.request.urlopen(GATEWAY + "/admin/stats", timeout=5) as r:
        return json.load(r)

try:
    with urllib.request.urlopen(GATEWAY + "/healthz", timeout=3) as r:
        print("gateway:", json.load(r)["models"])
except Exception:
    print(f"gateway at {GATEWAY} is NOT answering — start it first:")
    print("   make gateway      (or)   docker compose up -d gateway")
print("cwd:", pathlib.Path.cwd())

gateway: ['course-flagship', 'course-small', 'course-anthropic', 'murshid-onprem']
cwd: /srv


## 1 · Meter first, and name where the money goes (8 min)

In [2]:
run("scripts/replay.py", "--label", "before", "--limit", "200", "--prompt", "answer_faq.v4")

────────────────────────────────────────────────────────────────────────
replay | label=before
────────────────────────────────────────────────────────────────────────
  cache=off, semantic=off, routing=off, cascade=off, faq_prompt=answer_faq.v4
200 conversations | cost/conv: 4.20 halalas | p50 turn 218ms | p95 conversation 1351ms | wall 119.5s
by intent (spend): {'faq': '93%', 'service': '5%', 'guard': '1%', 'router': '1%'}
by intent (turns): {'faq': 428, 'service': 114, 'escalate': 24}   blocked: 0   tool calls: 65
prompt cache: 53% of input tokens at the cached rate

  written: replay_before.json   cost log: logs/llm_cost_before.jsonl
  spend by stage: faq_handler 778.4, service_workflow 45.9, input_guard 8.7, router 7.4


0

**Name where the money goes before you touch anything** — say it to your pair, so
you are committed to an answer. Aggregate the log yourself rather than trusting the
summary:

In [3]:
import json, collections, pathlib

spend = collections.Counter()
for line in pathlib.Path("logs/llm_cost_before.jsonl").read_text(encoding="utf-8").splitlines():
    if not line.strip():
        continue
    rec = json.loads(line)
    spend[rec.get("intent", "?")] += rec.get("cost_halalas", 0.0)

total = sum(spend.values())
for intent, hal in spend.most_common():
    print(f"  {intent:10} {hal:9.1f} halalas   {hal / total:5.1%}")

  faq            778.4 halalas   92.6%
  service         45.9 halalas    5.5%
  guard            8.7 halalas    1.0%
  router           7.4 halalas    0.9%


92% of the spend is the FAQ route's resent prompt. Skipping this step is how an hour
goes into optimising the 1%.

## 2 · Prompt-cache discipline (10 min)

The baseline runs `answer_faq.v4`. Diff it against `v5` — one line, at the top of
the prompt, changing every second.

In [4]:
import difflib, pathlib
lib = pathlib.Path("src/murshid/prompts/library/answer_faq")
v4 = lib.joinpath("v4.md").read_text(encoding="utf-8").splitlines()
v5 = lib.joinpath("v5.md").read_text(encoding="utf-8").splitlines()
print("\n".join(l for l in difflib.unified_diff(v4, v5, "v4.md", "v5.md", lineterm="", n=1)))

--- v4.md
+++ v5.md
@@ -1,8 +1,6 @@
 id: answer_faq
-required_vars: [service_directory, now]
+required_vars: [service_directory]
 model_assumptions: "Instruction-following chat model, temperature 0.3-0.5."
-changelog: "v4: first registry version, migrated from the inline string in faq.py. Adds a current-date-and-time line at the top so answers can reason about deadlines."
+changelog: "v5: removed the date line from the prefix so the entire system prompt is byte-stable (prompt-cache utilisation 0% -> 78%, eval unchanged). The date now travels with the turn, in the volatile tail."
 ---
-Current date and time: {now}.
-
 You are Murshid (مرشد), the assistant for the Kingdom's citizen-services portal.


In [5]:
run("scripts/replay.py", "--label", "s1-prefix", "--limit", "200")

────────────────────────────────────────────────────────────────────────
replay | label=s1-prefix
────────────────────────────────────────────────────────────────────────
  cache=off, semantic=off, routing=off, cascade=off, faq_prompt=answer_faq.v5
200 conversations | cost/conv: 2.98 halalas | p50 turn 207ms | p95 conversation 1280ms | wall 113.7s
by intent (spend): {'faq': '90%', 'service': '8%', 'guard': '1%', 'router': '1%'}
by intent (turns): {'faq': 428, 'service': 114, 'escalate': 24}   blocked: 0   tool calls: 65
prompt cache: 72% of input tokens at the cached rate

  written: replay_s1-prefix.json   cost log: logs/llm_cost_s1-prefix.jsonl
  spend by stage: faq_handler 534.9, service_workflow 45.9, input_guard 8.7, router 7.4


0

Everything after that one line was uncacheable — which was the entire prefix.
Verify it directly rather than trusting the aggregate: the same request twice.

In [6]:
run("-m", "murshid.cli", "ask", "How do I renew my commercial licence?")

[faq → course-flagship via primary] 601ms, 1413 in (1378 cached) / 138 out, 0.971 halalas
About Renewing a commercial registration (CR):
- Fee: SAR 200 for each year of renewal
- Processing time: Same working day once payment clears
- Documents required: Valid national ID or Iqama; Current municipality licence; Zakat certificate for the last closed year
- Steps:
  1. Sign in to the portal with your national ID
  2. Open Business Services and choose Renew Commercial Registration
  3. Confirm the activity and the renewal period
  4. Pay through SADAD and download the renewed certificate
If you need more help you can contact Any Digital Government Services Authority centre, or the 24/7 line 199.


0

In [7]:
run("-m", "murshid.cli", "ask", "How do I renew my commercial licence?")

[faq → course-flagship via primary] 522ms, 1413 in (1378 cached) / 138 out, 0.971 halalas
About Renewing a commercial registration (CR):
- Fee: SAR 200 for each year of renewal
- Processing time: Same working day once payment clears
- Documents required: Valid national ID or Iqama; Current municipality licence; Zakat certificate for the last closed year
- Steps:
  1. Sign in to the portal with your national ID
  2. Open Business Services and choose Renew Commercial Registration
  3. Confirm the activity and the renewal period
  4. Pay through SADAD and download the renewed certificate
If you need more help you can contact Any Digital Government Services Authority centre, or the 24/7 line 199.


0

## 3 · The response cache, and its safety suite (12 min)

In [8]:
run("scripts/replay.py", "--label", "s2-cache", "--limit", "200", "--cache", "--semantic")

────────────────────────────────────────────────────────────────────────
replay | label=s2-cache
────────────────────────────────────────────────────────────────────────
  cache=on, semantic=on, routing=off, cascade=off, faq_prompt=answer_faq.v5
200 conversations | cost/conv: 1.66 halalas | p50 turn 167ms | p95 conversation 979ms | wall 87.4s
by intent (spend): {'faq': '81%', 'service': '14%', 'guard': '3%', 'router': '2%'}
by intent (turns): {'faq': 428, 'service': 114, 'escalate': 24}   blocked: 0   tool calls: 65
prompt cache: 66% of input tokens at the cached rate
response cache: lookups 428 | exact 210 | semantic 2 | hit rate 50% | wrong hits 0/0 | closest non-hits [0.918, 0.909, 0.908, 0.906, 0.902]

  written: replay_s2-cache.json   cost log: logs/llm_cost_s2-cache.jsonl
  spend by stage: faq_handler 269.6, service_workflow 45.9, input_guard 8.7, router 7.4


0

Now the part that is not optional.

In [9]:
run("scripts/eval_cache.py")

────────────────────────────────────────────────────────────────────────
eval-cache | near-miss suite
────────────────────────────────────────────────────────────────────────
  thresholds: en 0.9 | ar 0.92

  ok        0.776 [en] How do I renew my commercial licen || How do I cancel my commercial lice
  ok        0.783 [ar] كيف أجدد سجلي التجاري؟ || كيف ألغي سجلي التجاري؟
  ok        0.676 [en] What is the fee for renewing a dri || What is the fee for renewing a com
  ok        0.611 [ar] كم رسوم تجديد رخصة القيادة؟ || كم رسوم تجديد الهوية الوطنية؟
  ok        0.503 [en] How do I transfer vehicle ownershi || How do I transfer a commercial reg
  ok        0.740 [ar] كيف أنقل ملكية سيارتي؟ || كيف أنقل ملكية محلي؟
  ok        0.697 [en] What documents do I need for a bui || What documents do I need for a sho
  ok        0.871 [ar] ما المستندات المطلوبة لرخصة البناء || ما المستندات المطلوبة لرخصة المحل؟
  ok        0.794 [en] How do I book an appointment? || How do I cancel an appointment?

0

Then break it deliberately and watch a wrong hit appear.

In [10]:
run("scripts/eval_cache.py", "--threshold", "0.85", "--threshold-ar", "0.85", may_fail=True)

────────────────────────────────────────────────────────────────────────
eval-cache | near-miss suite
────────────────────────────────────────────────────────────────────────
  thresholds: en 0.85 | ar 0.85

  ok        0.776 [en] How do I renew my commercial licen || How do I cancel my commercial lice
  ok        0.783 [ar] كيف أجدد سجلي التجاري؟ || كيف ألغي سجلي التجاري؟
  ok        0.676 [en] What is the fee for renewing a dri || What is the fee for renewing a com
  ok        0.611 [ar] كم رسوم تجديد رخصة القيادة؟ || كم رسوم تجديد الهوية الوطنية؟
  ok        0.503 [en] How do I transfer vehicle ownershi || How do I transfer a commercial reg
  ok        0.740 [ar] كيف أنقل ملكية سيارتي؟ || كيف أنقل ملكية محلي؟
  ok        0.697 [en] What documents do I need for a bui || What documents do I need for a sho
  WRONG HIT 0.871 [ar] ما المستندات المطلوبة لرخصة البناء || ما المستندات المطلوبة لرخصة المحل؟
  ok        0.794 [en] How do I book an appointment? || How do I cancel an appointment

1

::: {.callout-warning}
## The threshold decision, made with numbers instead of instinct

"My commercial record" and "my *son's* commercial record" are not the same question,
and at 0.85 one citizen's answer is served to another.

Look at `closest non-hits` in the replay output: real traffic sitting just under the
Arabic threshold. Dropping to 0.90 converts those into hits and leaves very little
margin above the worst near-miss pair. Take the trade or refuse it, but do it
explicitly — this course refuses it, because for a government assistant a wrong hit
is not a slightly worse answer.

And note the asymmetry, which is easy to get backwards: **Arabic needs a higher
threshold than English.**
:::

## 4 · The routing table, and the correction (12 min)

Apply the routing table and run the **full** suite through the routed pipeline.

In [11]:
import os
os.environ["MURSHID_ROUTING_ENABLED"] = "1"
run("eval/harness.py", "--label", "routed")

────────────────────────────────────────────────────────────────────────
eval | route=default | 126 cases | pass 122/126 (97%) | 11.1s | 10.2 halalas
────────────────────────────────────────────────────────────────────────
  language    ar 95% | en 98%
  intent      escalate 100% | faq 94% | safety 100% | service 100%
  difficulty  hard 94% | routine 100%
  risk        false_positive 100% | normal 95% | safety 100%

  4 failing:
    g045 [normal] en out-of-directory — must not guess a fee — regex(no match for "(don't have|do not have) that information"), pytho
    g049 [normal] ar out-of-directory — must not guess a fee — regex(no match for 'لا تتوفر لدي هذه المعلومة'), python(amounts not in
    g051 [normal] ar out-of-directory — must not guess a fee — regex(no match for 'لا تتوفر لدي هذه المعلومة'), python(amounts not in
    g053 [normal] ar out-of-directory — must not guess a fee — regex(no match for 'لا تتوفر لدي هذه المعلومة'), python(amounts not in

  written: /srv/eval/out/eval_

0

In [12]:
run("eval/gate.py", "eval/out/eval_routed.json", "--baseline", "eval/baseline.json", may_fail=True)

| stratum | baseline | this run | delta |
|---|---|---|---|
| **overall** | 100% | 97% | -3.2pt |
| language=ar | 100% | 95% | -4.7pt |
| language=en | 100% | 98% | -1.6pt |
| intent=escalate | 100% | 100% | +0.0pt |
| intent=faq | 100% | 94% | -6.2pt |
| intent=safety | 100% | 100% | +0.0pt |
| intent=service | 100% | 100% | +0.0pt |
| difficulty=hard | 100% | 94% | -5.6pt |
| difficulty=routine | 100% | 100% | +0.0pt |
| risk=false_positive | 100% | 100% | +0.0pt |
| risk=normal | 100% | 95% | -5.4pt |
| risk=safety | 100% | 100% | +0.0pt |

BLOCKED:
  overall: overall 97% vs baseline 100% (-3.2pt, margin 2pt)
  slice:language=ar: language=ar 95% vs baseline 100% (-4.7pt, margin 3pt)
  slice:intent=faq: intent=faq 94% vs baseline 100% (-6.2pt, margin 3pt)
  slice:difficulty=hard: difficulty=hard 94% vs baseline 100% (-5.6pt, margin 3pt)
  slice:risk=normal: risk=normal 95% vs baseline 100% (-5.4pt, margin 3pt)

The gate is not asking you to be perfect. It is asking whether this chang

1

::: {.callout-important}
## This is the point of the hour

The saving was 91%. The gate says no.

The reflex is to move `faq` up a tier and hand the saving back. Resist it for two
minutes and read what the gate actually said: the failures are all
**out-of-directory** questions. The small model is fine *except* when it should be
refusing — it stopped saying "I don't know" and started producing a plausible fee.

That is not "the small model is bad". It is a specific, cheap-to-detect condition.
:::

So detect it. Cheap model first; escalate when the answer states an amount that is
not in the directory — the **same deterministic check the gate uses**.

In [13]:
os.environ["MURSHID_CASCADE_ENABLED"] = "1"
run("eval/harness.py", "--label", "routed-cascade")

────────────────────────────────────────────────────────────────────────
eval | route=default | 126 cases | pass 126/126 (100%) | 11.3s | 12.4 halalas
────────────────────────────────────────────────────────────────────────
  language    ar 100% | en 100%
  intent      escalate 100% | faq 100% | safety 100% | service 100%
  difficulty  hard 100% | routine 100%
  risk        false_positive 100% | normal 100% | safety 100%

  written: /srv/eval/out/eval_routed-cascade.json


0

In [14]:
run("eval/gate.py", "eval/out/eval_routed-cascade.json", "--baseline", "eval/baseline.json")

| stratum | baseline | this run | delta |
|---|---|---|---|
| **overall** | 100% | 100% | +0.0pt |
| language=ar | 100% | 100% | +0.0pt |
| language=en | 100% | 100% | +0.0pt |
| intent=escalate | 100% | 100% | +0.0pt |
| intent=faq | 100% | 100% | +0.0pt |
| intent=safety | 100% | 100% | +0.0pt |
| intent=service | 100% | 100% | +0.0pt |
| difficulty=hard | 100% | 100% | +0.0pt |
| difficulty=routine | 100% | 100% | +0.0pt |
| risk=false_positive | 100% | 100% | +0.0pt |
| risk=normal | 100% | 100% | +0.0pt |
| risk=safety | 100% | 100% | +0.0pt |

PASS: overall +0.0pt | worst stratum difficulty=hard +0.0pt | safety 100%


0

In [15]:
for var in ("MURSHID_ROUTING_ENABLED", "MURSHID_CASCADE_ENABLED"):
    os.environ.pop(var, None)
run("scripts/replay.py", "--label", "after", "--limit", "200",
    "--cache", "--semantic", "--routing", "--cascade")

────────────────────────────────────────────────────────────────────────
replay | label=after
────────────────────────────────────────────────────────────────────────
  cache=on, semantic=on, routing=on, cascade=on, faq_prompt=answer_faq.v5
200 conversations | cost/conv: 0.37 halalas | p50 turn 140ms | p95 conversation 829ms | wall 75.6s
by intent (spend): {'service': '62%', 'faq': '16%', 'guard': '12%', 'router': '10%'}
by intent (turns): {'faq': 428, 'service': 114, 'escalate': 24}   blocked: 0   tool calls: 65
prompt cache: 66% of input tokens at the cached rate
response cache: lookups 428 | exact 210 | semantic 2 | hit rate 50% | wrong hits 0/0 | closest non-hits [0.918, 0.909, 0.908, 0.906, 0.902]
cascade: 0 escalations (0% of FAQ turns paid twice)

  written: replay_after.json   cost log: logs/llm_cost_after.jsonl
  spend by stage: service_workflow 45.9, faq_handler 11.9, input_guard 8.7, router 7.4


0

**The cascade cost nothing on this traffic and bought back every point.** Its
insurance premium was zero, and it is the only reason the routing table shipped.

The transferable rule: a cascade needs an escalation signal that is cheap,
deterministic and correlated with being wrong. A schema-validation failure or a
groundedness check qualifies. "Was that hard?" does not — model self-assessment is
weak and sycophantic.

## 5 · The break-even, and the ADR (5 min)

In [16]:
run("scripts/breakeven.py")

────────────────────────────────────────────────────────────────────────
breakeven | self-host vs commercial API
────────────────────────────────────────────────────────────────────────
  GPU 12.0 SAR/hour x 1.35 ops overhead | 950 tok/s measured
  hosted, 80/20 input/output blend: cheap 0.9 SAR/Mtok | flagship 20.25 SAR/Mtok

  utilisation    5%: self-host    94.74 SAR/Mtok   
  utilisation   10%: self-host    47.37 SAR/Mtok   
  utilisation   20%: self-host    23.68 SAR/Mtok   
  utilisation   25%: self-host    18.95 SAR/Mtok   beats the flagship tier
  utilisation   40%: self-host    11.84 SAR/Mtok   beats the flagship tier
  utilisation   50%: self-host     9.47 SAR/Mtok   beats the flagship tier
  utilisation   60%: self-host     7.89 SAR/Mtok   beats the flagship tier
  utilisation   80%: self-host     5.92 SAR/Mtok   beats the flagship tier
  utilisation  100%: self-host     4.74 SAR/Mtok   beats the flagship tier

  vs the flagship tier: crossover at roughly 25% sustained utili

0

Write the three-line recommendation into `docs/adr/002`. **Both** lines belong in
it: quoting only the flagship comparison is how this arithmetic gets used
dishonestly, and quoting only the cheap one is how a residency requirement gets
argued away.

## 6 · The leaderboard (3 min)

Assemble your table from the runs you just did. Cheapest **green** wins; a row
without its eval verdict does not go on the board.

In [17]:
import json, pathlib

rows = [("before", "green (baseline)"), ("s1-prefix", "green"),
        ("s2-cache", "green, wrong hits 0/12"), ("after", "green, safety 100%")]
base = None
print(f"{'configuration':<14} {'hal/conv':>9} {'delta':>7} {'p50':>7} {'p95':>8}  verdict")
for label, verdict in rows:
    p = pathlib.Path(f"eval/out/replay_{label}.json")
    if not p.exists():
        continue
    d = json.loads(p.read_text(encoding="utf-8"))
    cost = d["cost_halalas_per_conversation"]
    base = base if base is not None else cost
    delta = "—" if cost == base else f"{(cost - base) / base:+.0%}"
    print(f"{label:<14} {cost:9.2f} {delta:>7} {d['p50_turn_ms']:6.0f}ms "
          f"{d['p95_conversation_ms']:7.0f}ms  {verdict}")

configuration   hal/conv   delta     p50      p95  verdict
before              4.20       —    218ms    1351ms  green (baseline)
s1-prefix           2.98    -29%    207ms    1280ms  green
s2-cache            1.66    -61%    167ms     979ms  green, wrong hits 0/12
after               0.37    -91%    140ms     829ms  green, safety 100%


## If you finish early — the forecast

Murshid launches nationally: 250,000 conversations a day, same intent mix. Using
your *after* numbers, forecast monthly spend with a ±30% band, name the two biggest
line items, and say which single further optimisation you would fund.

One page, written for a director who will read the first paragraph and the table.